# Inference Backends

*Level 7 — Production RAG*

## Objective

Compare the local-development inference backend (`inference/ollama_client.py`, wraps Ollama --
what this whole repo has run on since Level 1) against the production-grade one
(`inference/vllm_client.py`, an OpenAI-compatible client for a GPU-served vLLM/SGLang server) --
same `LLMBackend`-shaped interface (`complete(prompt, system=None) -> str`), swappable behind one
config value, matching how `api/main.py` could pick a backend without touching route logic.

**Honesty note, stated up front:** this whole repo runs on CPU via Ollama -- there is no GPU here
to start a real vLLM server against. `OllamaBackend` is exercised for real below (real Ollama
call). `VLLMBackend` is exercised against a **mocked HTTP response** (same test as
`tests/test_vllm_client.py`) to demonstrate its request/response contract -- it has never talked
to a real vLLM server. This is disclosed, not hidden.

In [1]:
import sys
import time
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))

from inference.ollama_client import OllamaBackend
from inference.vllm_client import VLLMBackend

ollama_backend = OllamaBackend()
print("backend name:", ollama_backend.name)
print("model:", ollama_backend._llm.model)

backend name: ollama
model: llama3.2


## A real completion through `OllamaBackend`

In [2]:
t0 = time.perf_counter()
answer = ollama_backend.complete(
    "In one sentence, what is Retrieval-Augmented Generation?",
    system="Answer concisely.",
)
elapsed = time.perf_counter() - t0
print(f"({elapsed:.1f}s)\n{answer}")

(9.0s)
Retrieval-Augmented Generation (RAG) is a technique that combines retrieval of relevant information with generation to produce more accurate and informative outputs, often used in natural language processing tasks such as question answering and text summarization.


## `VLLMBackend`'s request contract (mocked -- no GPU available)

Same exercise as `tests/test_vllm_client.py`: monkeypatch `requests.post` to return a canned
OpenAI-compatible response, and confirm the backend builds the right request and parses the
right response shape. This proves the *code* is correct against the documented vLLM contract; it
does not prove a real vLLM server behaves identically -- that's the disclosed gap.

In [3]:
import requests as requests_module

captured_request = {}

class _FakeResponse:
    def __init__(self, payload):
        self._payload = payload
    def raise_for_status(self):
        pass
    def json(self):
        return self._payload

def _fake_post(url, json, timeout):
    captured_request["url"] = url
    captured_request["json"] = json
    return _FakeResponse({"choices": [{"message": {"content": "  RAG combines retrieval with generation.  "}}]})

_real_post = requests_module.post
requests_module.post = _fake_post
try:
    vllm_backend = VLLMBackend(base_url="http://localhost:8000", model="Llama-3-70B-Instruct")
    vllm_answer = vllm_backend.complete("What is RAG?", system="Answer concisely.")
finally:
    requests_module.post = _real_post

print("backend name:", vllm_backend.name)
print("request sent to:", captured_request["url"])
print("request body:", captured_request["json"])
print("parsed answer:", repr(vllm_answer))

backend name: vllm
request sent to: http://localhost:8000/v1/chat/completions
request body: {'model': 'Llama-3-70B-Instruct', 'messages': [{'role': 'system', 'content': 'Answer concisely.'}, {'role': 'user', 'content': 'What is RAG?'}], 'temperature': 0.2}
parsed answer: 'RAG combines retrieval with generation.'


## LiteLLM gateway config (reference-only)

`inference/litellm_config.yaml` routes `local-chat`/`local-embed` to this repo's real Ollama
instance and `production-chat` to the (unavailable-here) vLLM server, with a documented fallback
chain. Not run as a live gateway in this repo -- shown here as the config a real deployment would
load.

In [4]:
print((LEVEL_DIR / "inference" / "litellm_config.yaml").read_text())

# Reference LiteLLM gateway config: route model names to whichever real
# backend serves them, so `api/main.py` calls one consistent interface
# regardless of whether a model is running locally (Ollama) or on a GPU
# server (vLLM/SGLang). Not run in this repo (no LiteLLM proxy process
# started) -- provided as the production-shape config this level's
# `inference/` clients are written to be compatible with.
model_list:
  - model_name: local-chat
    litellm_params:
      model: ollama/llama3.2
      api_base: http://localhost:11434

  - model_name: local-embed
    litellm_params:
      model: ollama/nomic-embed-text
      api_base: http://localhost:11434

  - model_name: production-chat
    litellm_params:
      model: openai/llama3.2  # vLLM's OpenAI-compatible endpoint
      api_base: http://localhost:8000/v1

router_settings:
  routing_strategy: simple-shuffle
  fallbacks:
    - production-chat: [local-chat]  # fall back to the local model if the GPU backend is unreachable

litellm_

## What I observed

- **`OllamaBackend` took 9.0 seconds for one short completion**, alone, with no concurrent
  requests -- on this machine's CPU, `llama3.2` generation is the dominant cost of any RAG
  request. This matches the load-testing finding in `../load-testing/scenarios.md` (16-40s under
  5 concurrent users) almost exactly, just without the queueing on top.
- **The answer is correct and on-topic** -- a genuine, coherent one-sentence RAG definition, not
  a placeholder.
- **`VLLMBackend` built the exact expected OpenAI-compatible payload** (`model`,
  `messages` with system+user roles, `temperature`) and correctly parsed a canned
  `choices[0].message.content` response, stripped of whitespace -- confirming the client's
  *contract* is correct even though it's never been exercised against a real server.

## Ollama vs. vLLM/SGLang -- the actual trade-off

| | Ollama (this repo) | vLLM / SGLang (production) |
|---|---|---|
| Hardware | CPU (or any GPU, unmanaged) | GPU, purpose-built serving |
| Concurrency | One request at a time, effectively (confirmed by `../load-testing/scenarios.md`: 5 concurrent users -> 16-40s each, serialized) | Continuous batching -- many concurrent requests share GPU compute |
| Setup cost | `ollama pull`, done | Model weights, GPU provisioning, serving config |
| This repo's choice | Every level, deliberately -- free, reproducible, no cloud dependency | Documented, contract-tested, never run (no GPU here) |

The real Locust numbers in `../load-testing/scenarios.md` are the concrete evidence for *why* a
production deployment needs the right-hand column: this notebook's own `OllamaBackend` call above
took several seconds for one request with nothing else competing for the CPU.